# MiniMax H3 Singularity — Kaggle T4 x2 레퍼런스 투 비디오 (R2V)

레퍼런스 이미지(캐릭터 시트, 배경 등)와 프롬프트로 영상을 만드는 노트북이에요.
ComfyUI를 화면 없이 설치해 API로 실행하고, 모델은 임시 폴더(`/tmp`)에 받아요.

**T4 기준 기본값:** 0.2MP(608x352), 2초, 8스텝 1패스. 원본 RunningHub 설정(0.5MP, 10초, 12스텝 2패스 + 업스케일)은 T4에서 사실상 불가능해요.

**준비 (Kaggle 노트북 오른쪽 패널)**
1. Settings → Accelerator: **GPU T4 x2**
2. Settings → Internet: **On**
3. Add Input → 이미지를 올린 **Dataset**을 추가하고, 설정 셀의 경로를 맞춰요.
4. (선택) Add-ons → Secrets에 `HF_TOKEN`을 추가하면 다운로드가 안정적이에요.

**실행:** 위에서부터 순서대로 실행해요. 결과 영상은 `/kaggle/working/outputs`에 저장돼요.
설정만 바꿔 다시 만들 때는 설정 셀과 **5번 셀만** 다시 실행하면 돼요(모델과 서버는 그대로 재사용).

In [ ]:
# ============ 설정 (여기만 고치면 돼요) ============

# 영상 모델: "gguf" (T4 권장, 어떤 GPU에서도 동작) | "w4a8" (11.8GB, T4 지원 불확실) | "int8_pruned" (21GB)
MODEL_VARIANT = "gguf"
GGUF_PREFERENCE = ["Q4_K_M", "Q4_K_S", "Q4_0", "Q5_K_M", "Q3_K_M"]  # 앞에 있는 것부터 찾아요

# 텍스트 인코더: "nvfp4_awq" (약 15GB, T4 권장) | "int8_convrot" (약 25GB, T4에 너무 커요)
TEXT_ENCODER = "nvfp4_awq"
TE_DEVICE = "cuda:1"          # 텍스트 인코더를 두 번째 T4에 올려요. 문제가 생기면 "default"

TURBO_LORA = True             # lightx2v 4스텝 Turbo LoRA (Singularity 제작자 권장)

# 레퍼런스 이미지: 순서대로 프롬프트의 <Picture 1>, <Picture 2>, ... (최대 9장)
REF_IMAGES = [
    "/kaggle/input/my-refs/character.png",
]

# 레퍼런스 영상 (선택): 동작·카메라를 따라갈 영상. 순서대로 프롬프트의 <Video 1>, <Video 2>, <Video 3>
# 최대 3개, 각 2~15초, 합계 15초 이하. 뎁스 노트북의 결과(컬러/흑백 뎁스)를 넣으면 동작만 가져와요.
REF_VIDEOS = [
    # "/kaggle/input/video-to-depth/depth/action_depth_color_1010_1200.mp4",
]
REF_VIDEO_AUDIO = False       # True: 레퍼런스 영상의 소리도 함께 참고 (대사·리듬 타이밍)

PROMPT = """subject_definitions:
<Subject 1> is the character in <Picture 1>.
<Picture 1> is the character design reference sheet for <Subject 1>.

summary:
[reference generation] A short cinematic shot of <Subject 1>.

retention_analysis:
<Subject 1> (appears in [Shot 1]): fully_preserved - face, hair, and costume from <Picture 1> are fully preserved.

detailed_description:
The target video uses a photorealistic cinematic style with natural lighting.
[Shot 1] A medium shot of <Subject 1> slowly turning toward the camera, wind moving the hair and clothes.

overall_soundscape:
Soft wind ambience.

non_diegetic_music:
N/A"""
# 레퍼런스 영상을 쓸 때는 subject_definitions에 이렇게 적어요:
# <Subject 1> is the character whose motion comes from <Video 1> and whose appearance comes from <Picture 1>.
# <Video 1> is the motion and camera reference (a depth video) for the target video.

SECONDS = 2.0                 # 영상 길이(초). T4에서는 2~3초 권장. 레퍼런스 영상과 비슷하게 맞추세요
MEGAPIXELS = 0.2              # 0.2 = 608x352 (16:9). 0.3 = 736x416
ASPECT = "16:9"               # "16:9" | "9:16" | "1:1"
FPS = 24
STEPS = 8                     # Turbo LoRA 기준 6~8 권장
TWO_PASS = False              # True: 원본처럼 12스텝 스케줄을 SPLIT_AT에서 나눠 2번 샘플링 (업스케일 없이)
SPLIT_AT = 8                  # TWO_PASS일 때만 사용 (원본: 12스텝 중 8)
SEED = 42

EXTRA_NODE_PACKS = {}
INPUT_FILES = REF_IMAGES + REF_VIDEOS

# T4는 bf16을 지원하지 않아서 fp16으로 강제해요. 영상이 검게 나오면 "--force-fp32"로 바꿔 보세요(느려요).
DTYPE_FLAG = "--force-fp16"
COMFY_ARGS = ["--lowvram", "--disable-smart-memory", "--cache-none", "--reserve-vram", "1.0", DTYPE_FLAG]

COMFY = "/tmp/ComfyUI"
MODEL_DIR = "/tmp/models"
OUT_DIR = "/kaggle/working/outputs"

## 1. 환경 확인

In [ ]:
import os, shutil, subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])
total, used, free = shutil.disk_usage("/tmp")
print(f"/tmp 여유 공간: {free / 1e9:.0f} GB (모델에 약 30GB 필요)")
with open("/proc/meminfo") as f:
    mem = {l.split(":")[0]: int(l.split()[1]) for l in f}
print(f"RAM: {mem['MemTotal'] / 1e6:.0f} GB, 사용 가능 {mem['MemAvailable'] / 1e6:.0f} GB")
if free < 32e9:
    print("⚠ /tmp 공간이 부족할 수 있어요. 다운로드가 실패하면 MODEL_VARIANT를 더 작은 것으로 바꾸세요.")
missing = [p for p in INPUT_FILES if not os.path.isfile(p)]
if missing:
    print("⚠ 입력 이미지를 찾을 수 없어요:", missing)
    print("  /kaggle/input 안의 파일:")
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files[:20]:
            print("   ", os.path.join(root, fn))

## 2. ComfyUI와 커스텀 노드 설치

In [ ]:
import subprocess, sys, os

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.isdir(COMFY):
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}")
else:
    sh(f"git -C {COMFY} pull --ff-only")

# 검증된 노드 팩만 사용해요 (출처가 불분명한 노드 팩은 설치하지 않아요)
NODE_PACKS = {
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes",      # MiniMax 저VRAM 패치
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF",           # GGUF 모델 로더
    "ComfyUI-MultiGPU": "https://github.com/pollockjj/ComfyUI-MultiGPU", # 텍스트 인코더를 두 번째 GPU로
    **EXTRA_NODE_PACKS,
}
for name, url in NODE_PACKS.items():
    path = f"{COMFY}/custom_nodes/{name}"
    if not os.path.isdir(path):
        sh(f"git clone --depth 1 {url} {path}")

sh(f"{sys.executable} -m pip install -q -r {COMFY}/requirements.txt")
for name in NODE_PACKS:
    req = f"{COMFY}/custom_nodes/{name}/requirements.txt"
    if os.path.isfile(req):
        sh(f"{sys.executable} -m pip install -q -r {req}")
sh(f"{sys.executable} -m pip install -q -U huggingface_hub gguf")

import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.device_count())

## 3. 모델 다운로드 (`/tmp/models`, 세션이 끝나면 지워져요)

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

api = HfApi(token=token)

def pick(repo, includes, prefer=(), ext=".safetensors"):
    files = [f for f in api.list_repo_files(repo) if f.endswith(ext)]
    hits = [f for f in files if all(s.lower() in f.lower() for s in includes)]
    for p in prefer:
        for f in hits:
            if p.lower() in f.lower():
                return f
    if hits:
        return hits[0]
    raise FileNotFoundError(f"{repo}에서 {includes} 파일을 못 찾았어요. 파일 목록:\n  " + "\n  ".join(files))

SINGULARITY = "WarmBloodAban/Minimax-h3_Singularity"
SINGULARITY_GGUF = "Abiray/MiniMax-H3-Singularity-GGUF"
COMFY_ORG = "Comfy-Org/MiniMax-H3"
TURBO = "lightx2v/Minimax-h3-Turbo"

plan = {}  # 종류 -> (저장소, 파일, ComfyUI 모델 폴더)
if MODEL_VARIANT == "gguf":
    plan["unet"] = (SINGULARITY_GGUF, pick(SINGULARITY_GGUF, [], GGUF_PREFERENCE, ext=".gguf"), "unet")
elif MODEL_VARIANT == "w4a8":
    plan["unet"] = (SINGULARITY, pick(SINGULARITY, ["w4a8"]), "diffusion_models")
elif MODEL_VARIANT == "int8_pruned":
    plan["unet"] = (SINGULARITY, pick(SINGULARITY, ["pruned", "int8"]), "diffusion_models")
else:
    raise ValueError(MODEL_VARIANT)
te_key = {"nvfp4_awq": ["qwen3vl", "nvfp4"], "int8_convrot": ["qwen3vl", "int8"]}[TEXT_ENCODER]
plan["te"] = (COMFY_ORG, pick(COMFY_ORG, te_key), "text_encoders")
plan["vae"] = (COMFY_ORG, pick(COMFY_ORG, ["video_vae", "fp16"]), "vae")
plan["avae"] = (COMFY_ORG, pick(COMFY_ORG, ["audio_vae"]), "vae")
if TURBO_LORA:
    plan["lora"] = (TURBO, pick(TURBO, ["ref2v", "4step", "comfyui"], ["v0.1"]), "loras")


FILES = {}
for key, (repo, fname, folder) in plan.items():
    print(f"⬇ {key}: {repo}/{fname}")
    local = hf_hub_download(repo, fname, local_dir=f"{MODEL_DIR}/{repo.replace('/', '__')}", token=token)
    os.makedirs(f"{COMFY}/models/{folder}", exist_ok=True)
    link = f"{COMFY}/models/{folder}/{os.path.basename(fname)}"
    if not os.path.exists(link):
        os.symlink(local, link)
    FILES[key] = os.path.basename(fname)
    print(f"   {os.path.getsize(local) / 1e9:.1f} GB → models/{folder}/{FILES[key]}")
print(FILES)

## 4. ComfyUI 서버 시작

In [ ]:
import subprocess, sys, time, json, urllib.request

BASE = "http://127.0.0.1:8188"
LOG = "/tmp/comfyui.log"

def server_up():
    try:
        urllib.request.urlopen(BASE + "/system_stats", timeout=5)
        return True
    except Exception:
        return False

if not server_up():
    server = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", *COMFY_ARGS],
        cwd=COMFY, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(180):
        if server_up() or server.poll() is not None:
            break
        time.sleep(5)

def log_tail(n=40):
    with open(LOG, errors="replace") as f:
        print("".join(f.readlines()[-n:]))

if server_up():
    print("✅ ComfyUI 실행 중")
    log_tail(15)
else:
    print("❌ ComfyUI가 시작되지 않았어요. 로그:")
    log_tail(80)

## 5. 워크플로우 도구 준비

In [ ]:
import json, math, os, shutil, time, uuid, urllib.request, urllib.error

def api(path, data=None):
    body = json.dumps(data).encode() if data is not None else None
    req = urllib.request.Request(BASE + path, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())

INFO = api("/object_info")

def combo_options(spec):
    t = spec[0]
    if isinstance(t, list):
        return t
    if t == "COMBO" and len(spec) > 1 and isinstance(spec[1], dict):
        return spec[1].get("options", [])
    return None

class Graph:
    """ComfyUI API 형식 그래프. 서버의 노드 정의로 입력 이름과 값을 검사해요."""
    def __init__(self):
        self.nodes, self.n = {}, 0

    def add(self, cls, **inputs):
        if cls not in INFO:
            raise RuntimeError(f"서버에 '{cls}' 노드가 없어요. 설치 셀 로그를 확인하세요.")
        req = INFO[cls]["input"].get("required", {})
        specs = {**req, **INFO[cls]["input"].get("optional", {})}
        for name, spec in req.items():
            if name in inputs or any(k.startswith(name + ".") for k in inputs):
                continue
            cfg = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
            opts = combo_options(spec)
            if opts:
                inputs[name] = cfg.get("default", opts[0])
            elif "default" in cfg:
                inputs[name] = cfg["default"]
            else:
                raise RuntimeError(f"{cls}: 필수 입력 '{name}'이 비어 있어요. 정의: {spec}")
        for name, value in inputs.items():
            if name not in specs:
                print(f"  · {cls}: 정의에 없는 입력 '{name}' (동적 입력이면 정상)")
                continue
            opts = combo_options(specs[name])
            if opts and isinstance(value, str) and value not in opts:
                raise RuntimeError(f"{cls}.{name}='{value}'가 선택지에 없어요: {opts[:15]}")
        self.n += 1
        nid = str(self.n)
        self.nodes[nid] = {"class_type": cls, "inputs": inputs}
        return nid

def out(nid, i=0):
    return [nid, i]

def video_size(aspect, megapixels):
    """원본 워크플로우의 해상도 표와 같은 크기 (32의 배수)."""
    ar = {"16:9": 16 / 9, "9:16": 9 / 16, "1:1": 1.0}[aspect]
    w = math.ceil(math.sqrt(megapixels * 1e6 * ar) / 32) * 32
    h = math.ceil(math.sqrt(megapixels * 1e6 / ar) / 32) * 32
    return w, h

def frame_count(seconds):
    """원본 워크플로우의 프레임 수 계산식."""
    f = max(5, round(seconds * 24))
    return f + (5 - f % 17) % 17

def add_input_file(g, path, name, node="LoadImage", field="image"):
    os.makedirs(f"{COMFY}/input", exist_ok=True)
    fname = f"{name}{os.path.splitext(path)[1]}"
    shutil.copy(path, f"{COMFY}/input/{fname}")
    INFO[node] = api(f"/object_info/{node}")[node]  # 새 파일이 선택지에 보이도록 갱신
    return out(g.add(node, **{field: fname}))

def add_input_image(g, path, name):
    return add_input_file(g, path, name)

def load_models(g, shift=None):
    """영상 모델, 텍스트 인코더, VAE를 불러오고 T4용 패치와 LoRA를 적용해요."""
    if MODEL_VARIANT == "gguf":
        model = out(g.add("UnetLoaderGGUF", unet_name=FILES["unet"]))
    else:
        model = out(g.add("UNETLoader", unet_name=FILES["unet"], weight_dtype="default"))
    if TE_DEVICE != "default" and "CLIPLoaderMultiGPU" in INFO:
        clip = out(g.add("CLIPLoaderMultiGPU", clip_name=FILES["te"], type="minimax", device=TE_DEVICE))
    else:
        if TE_DEVICE != "default":
            print("⚠ CLIPLoaderMultiGPU가 없어서 텍스트 인코더를 기본 GPU에 올려요.")
        clip = out(g.add("CLIPLoader", clip_name=FILES["te"], type="minimax"))
    vae = out(g.add("VAELoader", vae_name=FILES["vae"]))
    audio_vae = out(g.add("VAELoader", vae_name=FILES["avae"]))
    # 원본의 SageAttention/SolAttention은 T4에서 지원되지 않아 KJNodes 저VRAM 패치로 대신해요.
    if "MiniMaxChunkFeedForward" in INFO:
        model = out(g.add("MiniMaxChunkFeedForward", model=model, chunks=4, seq_threshold=2048))
    if "MiniMaxLowVRAMAttention" in INFO:
        model = out(g.add("MiniMaxLowVRAMAttention", model=model, head_chunks=4))
    if shift:
        model = out(g.add("MiniMaxH3SigmaShift", model=model, shift_video=shift[0], shift_audio=shift[1]))
    if TURBO_LORA:
        lora = g.add("LoraLoader", model=model, clip=clip, lora_name=FILES["lora"],
                     strength_model=1.0, strength_clip=1.0)
        model, clip = out(lora, 0), out(lora, 1)
    return model, clip, vae, audio_vae

def save_video(g, images, audio, fps, prefix):
    video = out(g.add("CreateVideo", images=images, fps=float(fps), audio=audio))
    g.add("SaveVideo", video=video, filename_prefix=prefix)

def run(g):
    with open("/kaggle/working/last_workflow_api.json", "w") as fp:
        json.dump(g.nodes, fp, indent=1, ensure_ascii=False)
    try:
        prompt_id = api("/prompt", {"prompt": g.nodes, "client_id": uuid.uuid4().hex})["prompt_id"]
    except urllib.error.HTTPError as e:
        print("❌ 워크플로우 오류:")
        print(json.dumps(json.loads(e.read()), indent=1, ensure_ascii=False)[:4000])
        raise
    start = time.time()
    while True:
        hist = api(f"/history/{prompt_id}")
        if prompt_id in hist:
            status = hist[prompt_id].get("status", {})
            if status.get("status_str") == "error":
                print("\n❌ 실행 중 오류:")
                for m in status.get("messages", []):
                    if m[0] == "execution_error":
                        print(m[1].get("node_type"), "-", m[1].get("exception_message"))
                log_tail(40)
                raise RuntimeError("generation failed")
            break
        time.sleep(15)
        lines = open(LOG, errors="replace").read().replace("\r", "\n").strip().splitlines()
        print(f"\r{(time.time() - start) / 60:.1f}분 경과 | {lines[-1][:110] if lines else ''}", end="", flush=True)
    print(f"\n✅ 완료: {(time.time() - start) / 60:.1f}분")

print(f"노드 정의 {len(INFO)}개 불러옴")

## 6. 영상 만들기

In [ ]:
W, H = video_size(ASPECT, MEGAPIXELS)
LENGTH = frame_count(SECONDS)
print(f"해상도 {W}x{H}, {LENGTH}프레임")

g = Graph()
model, clip, vae, audio_vae = load_models(g)
refs = {f"ref_images.ref_image_{i}": add_input_image(g, p, f"ref_{i}") for i, p in enumerate(REF_IMAGES[:9])}
for i, p in enumerate(REF_VIDEOS[:3]):
    parts = g.add("GetVideoComponents", video=add_input_file(g, p, f"refvid_{i}", node="LoadVideo", field="file"))
    refs[f"ref_videos.ref_video_{i}"] = out(parts, 0)
    if REF_VIDEO_AUDIO:
        refs[f"ref_video_audios.ref_video_audio_{i}"] = out(parts, 1)
r2v = g.add("MiniMaxH3ReferenceToVideo", clip=clip, vae=vae, audio_vae=audio_vae,
            prompt=PROMPT, width=W, height=H, length=LENGTH, **refs)

noise = out(g.add("RandomNoise", noise_seed=SEED))
guider = out(g.add("BasicGuider", model=model, conditioning=out(r2v, 0)))
sampler = out(g.add("KSamplerSelect", sampler_name="euler"))
if TWO_PASS:
    sched = out(g.add("BasicScheduler", model=model, scheduler="beta", steps=12, denoise=1.0))
    split = g.add("SplitSigmas", sigmas=sched, step=SPLIT_AT)
    s1 = g.add("SamplerCustomAdvanced", noise=noise, guider=guider, sampler=sampler,
               sigmas=out(split, 0), latent_image=out(r2v, 1))
    s2 = g.add("SamplerCustomAdvanced", noise=noise, guider=guider, sampler=sampler,
               sigmas=out(split, 1), latent_image=out(s1, 1))
    latent = out(s2, 1)
else:
    sched = out(g.add("BasicScheduler", model=model, scheduler="beta", steps=STEPS, denoise=1.0))
    s1 = g.add("SamplerCustomAdvanced", noise=noise, guider=guider, sampler=sampler,
               sigmas=sched, latent_image=out(r2v, 1))
    latent = out(s1, 1)

images = out(g.add("VAEDecode", samples=latent, vae=vae))
audio = out(g.add("VAEDecodeAudio", samples=latent, vae=audio_vae))
PREFIX = f"singularity/r2v_{int(time.time())}"
save_video(g, images, audio, FPS, PREFIX)
run(g)

## 7. 결과 보기

In [ ]:
import glob, os, shutil
from IPython.display import Video, display

os.makedirs(OUT_DIR, exist_ok=True)
outs = sorted(glob.glob(f"{COMFY}/output/{PREFIX}*"), key=os.path.getmtime)
for path in outs:
    dest = shutil.copy(path, OUT_DIR)
    print("저장:", dest)
    display(Video(dest, embed=True, width=640))
if not outs:
    print("결과 파일이 없어요. 로그:")
    log_tail(40)

## 문제 해결

오류가 나면 **오류 메시지와 로그 마지막 부분**을 그대로 복사해서 Claude에게 붙여 주세요.
실행한 워크플로우는 `/kaggle/working/last_workflow_api.json`에 저장돼요.

| 증상 | 시도해 볼 것 |
|---|---|
| `CUDA out of memory` | `SECONDS`나 `MEGAPIXELS`를 낮추기, `GGUF_PREFERENCE`를 `Q3_K_M` 우선으로 |
| 커널이 죽음 (RAM 부족) | `TE_DEVICE="cuda:1"` 유지, 더 작은 GGUF 사용 |
| 영상이 검정/노이즈 | `DTYPE_FLAG="--force-fp32"` (느려짐) |
| `서버에 '...' 노드가 없어요` | 2번 셀 로그에서 해당 노드 팩 설치 오류 확인 |
| 텍스트 인코더 단계가 매우 느림 | NVFP4는 T4에서 에뮬레이션이라 느려요. 정상일 수 있어요 |